In [1]:
import CalculatedFieldSubroutines as cfs

#

import numpy as np

import pandas as pd

#

import matplotlib.pyplot as plt

from pandasgui import show

#

import warnings

#

import os

In [2]:
warnings.filterwarnings( 'ignore' )

In [3]:
gmIDs = cfs.list_whitelisted_gmIDs()

topics = cfs.list_topics()

print( topics )

['/apollo/sensor/gnss/best/pose', '/apollo/drive/event', '/apollo/canbus/chassis', '/apollo/perception/traffic/light']


In [4]:
def CreatePreprocessedMovingDataFolder( moving_window, expansion_window = 1 ): # sec

    expansion_window_ns = expansion_window * 1e9

    moving_window_ns = moving_window * 1e9

    #

    for index, gmID in enumerate( gmIDs ):

        chassis_df = cfs.retrieve_gmID_topic( gmID, '/apollo/canbus/chassis' )

        pose_df = cfs.retrieve_gmID_topic( gmID, '/apollo/sensor/gnss/best/pose' )

        #

        chassis_df = chassis_df.sort_values( 'time' )

        pose_df = pose_df.sort_values( 'time' )

        #

        cfs.Index( chassis_df )

        #

        cfs.NormalizedTime( chassis_df )

        #

        cfs.BinaryDrivingMode( chassis_df )

        cfs.TernaryDrivingModeTransition( chassis_df )

        cfs.BinaryDisengagement( chassis_df )

        cfs.BinaryDisengagementExpanded( chassis_df, moving_colname = 'time', window = expansion_window_ns )

        cfs.DisengagementID( chassis_df, expanded = False )

        cfs.DisengagementID( chassis_df, expanded = True )

        #

        cfs.Acceleration( chassis_df )

        #

        chassis_df = chassis_df.drop( [ 'drivingMode', 'TernaryDrivingModeTransition', 'signal.turnSignal' ], axis = 1 )

        #

        cfs.LatLonTotalStdDev( pose_df )

        #

        cfs.ProgressAlongRoute_v2( pose_df )

        #

        pose_df = pose_df.drop( [ 'heightMsl', 'groupMetadataID', 'latitudeStdDev', 'heightStdDev', 'longitudeStdDev', \
                                  'PartitionNumber' ], axis = 1 )

        #

        cfs.ChassisBestPoseMatchedTime( chassis_df, pose_df )

        merged_df = pd.merge( chassis_df, pose_df, on = 'ChassisBestPoseMatchedTime', how = 'inner' )

        merged_df = merged_df.rename( columns = { 'time_x' : 'time' } )

        merged_df = merged_df.drop( [ 'ChassisBestPoseMatchedTime', 'time_y' ], axis = 1 )

        #

        merged_df = merged_df[ [ 'Ind', 'groupMetadataID', 'time', 'NormalizedTime', 'BinaryDrivingMode', 'BinaryDisengagement', \
                                 'BinaryDisengagementExpanded', 'DisengagementID', 'DisengagementExpandedID', 'latitude', \
                                 'longitude', 'ProgressAlongRoute', 'solStatus', 'solType', 'extendedSolutionStatus', \
                                 'numSatsInSolution', 'speedMps', 'Acceleration', 'brakePercentage', 'throttlePercentage', \
                                 'steeringPercentage', 'LatLonTotalStdDev' ] ]

        #

        if ( moving_window > 0 ):

            cfs.MovingFunction_v2( merged_df, 'time', moving_window_ns, 'mean', [ 'speedMps', 'Acceleration', 'brakePercentage', \
                                                                                  'throttlePercentage', 'steeringPercentage', \
                                                                                  'LatLonTotalStdDev' ] )

            merged_df = merged_df.drop( [ 'speedMps', 'Acceleration', 'brakePercentage', 'throttlePercentage', \
                                          'steeringPercentage', 'LatLonTotalStdDev' ], axis = 1 )

        #

        filepath = f'{ cfs.origin_dir() }/Preprocessed_Moving_Data_v2/{ moving_window }sec_moving_window/{ gmID }'

        os.makedirs( filepath, exist_ok = True )

        #

        merged_df.to_csv( f'{ filepath }/{ gmID }.csv', index = False )

In [5]:
for sec in range( 0, 6 ):

    CreatePreprocessedMovingDataFolder( moving_window = sec )

    print( f'{ sec } second moving window data done' )

KeyboardInterrupt: 

In [ ]:
# if successful, this code should run

In [ ]:
test_df1 = cfs.retrieve_gmID_preprocessed_moving_data_v2( gmID = gmIDs[ 0 ], moving_window = 0 )

In [ ]:
test_df2 = cfs.retrieve_gmID_preprocessed_moving_data_v2( gmID = gmIDs[ 0 ], moving_window = 5 )

In [ ]:
plt.plot( test_df1[ 'NormalizedTime' ], test_df1[ 'speedMps' ], label = 'instant data' )

plt.plot( test_df2[ 'NormalizedTime' ], test_df2[ 'speedMps_mean' ], label = '5 sec moving average' )

plt.xlim( 0.4, 0.5 )

plt.legend()

plt.show()